# EDA — My Content
Exploration des données : articles, interactions utilisateurs, embeddings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import zipfile
import os

sns.set_theme(style='whitegrid')
DATA_DIR = '../data'

## 1. Articles metadata

In [ ]:
meta = pd.read_csv(f'{DATA_DIR}/articles_metadata.csv')
print(meta.shape)
meta.head()

In [ ]:
meta.info()

In [ ]:
meta.describe()

In [ ]:
print('Valeurs manquantes :')
print(meta.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

meta['category_id'].value_counts().head(20).plot(kind='bar', ax=axes[0], title='Top 20 catégories')
meta['publisher_id'].value_counts().head(20).plot(kind='bar', ax=axes[1], title='Top 20 éditeurs')
meta['words_count'].clip(upper=2000).plot(kind='hist', bins=50, ax=axes[2], title='Distribution mots par article')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution temporelle des articles
meta['created_at'] = pd.to_datetime(meta['created_at_ts'], unit='ms')
meta['created_at'].dt.year.value_counts().sort_index().plot(kind='bar', title='Articles par année de création')
plt.show()

## 2. Clicks (sample)

In [ ]:
clicks = pd.read_csv(f'{DATA_DIR}/clicks_sample.csv')
print(clicks.shape)
clicks.head()

In [ ]:
clicks.info()

In [ ]:
print('Valeurs manquantes :')
print(clicks.isnull().sum())

In [ ]:
print(f"Users uniques     : {clicks['user_id'].nunique()}")
print(f"Sessions uniques  : {clicks['session_id'].nunique()}")
print(f"Articles uniques  : {clicks['click_article_id'].nunique()}")
print(f"Interactions total: {len(clicks)}")

In [ ]:
# Clics par user
clicks_per_user = clicks.groupby('user_id').size()
print(clicks_per_user.describe())
clicks_per_user.clip(upper=20).plot(kind='hist', bins=20, title='Distribution clics par utilisateur')
plt.xlabel('Nombre de clics')
plt.show()

In [ ]:
# Articles les plus cliqués
clicks['click_article_id'].value_counts().head(20).plot(kind='bar', title='Top 20 articles cliqués')
plt.show()

In [ ]:
# Taille des sessions
clicks['session_size'].value_counts().sort_index().plot(kind='bar', title='Distribution taille des sessions')
plt.show()

In [ ]:
# Overlap articles cliqués vs articles dans metadata
clicked_articles = set(clicks['click_article_id'].unique())
meta_articles = set(meta['article_id'].unique())
overlap = clicked_articles & meta_articles
print(f"Articles cliqués dans le sample : {len(clicked_articles)}")
print(f"Articles dans metadata          : {len(meta_articles)}")
print(f"Overlap                         : {len(overlap)} ({len(overlap)/len(clicked_articles)*100:.1f}% des cliqués)")

## 3. Clicks (dataset complet)

In [ ]:
with zipfile.ZipFile(f'{DATA_DIR}/clicks.zip', 'r') as z:
    print('Fichiers dans le zip :', z.namelist())

In [ ]:
with zipfile.ZipFile(f'{DATA_DIR}/clicks.zip', 'r') as z:
    with z.open(z.namelist()[0]) as f:
        clicks_full = pd.read_csv(f)

print(clicks_full.shape)
print(f"Users uniques     : {clicks_full['user_id'].nunique()}")
print(f"Articles uniques  : {clicks_full['click_article_id'].nunique()}")
clicks_full.head()

In [ ]:
# Matrice de sparsité user-article
n_users = clicks_full['user_id'].nunique()
n_articles = clicks_full['click_article_id'].nunique()
n_interactions = len(clicks_full)
sparsity = 1 - (n_interactions / (n_users * n_articles))
print(f"Users    : {n_users}")
print(f"Articles : {n_articles}")
print(f"Interactions : {n_interactions}")
print(f"Sparsité : {sparsity:.4%}")

## 4. Embeddings articles

In [ ]:
with open(f'{DATA_DIR}/articles_embeddings.pickle', 'rb') as f:
    embeddings = pickle.load(f)

print(type(embeddings))
print(f"Shape : {embeddings.shape}")
print(f"Dtype : {embeddings.dtype}")

In [ ]:
# Taille en mémoire
size_mb = embeddings.nbytes / 1024**2
print(f"Taille en mémoire : {size_mb:.1f} MB")
print(f"Nb articles avec embeddings : {embeddings.shape[0]}")
print(f"Dimension des embeddings   : {embeddings.shape[1]}")

In [ ]:
# Overlap embeddings vs metadata
print(f"Articles dans metadata   : {len(meta_articles)}")
print(f"Articles dans embeddings : {embeddings.shape[0]}")
# Si l'index des embeddings correspond aux article_id
# (à vérifier selon la structure exacte)

## 5. Conclusions EDA

_(À compléter après exécution)_

- **Cold start users** : ...
- **Cold start articles** : ...
- **Sparsité** : ...
- **ACP nécessaire** : embeddings de ... dimensions → réduire à ...